# Introduction

Lucic-Tse Quoting Engine Validation

This notebook explores the `LucicTseQuotingEngine` with real data from the `PricingEngine`.
It visualises:
- Spreads vs. inventory level (asymmetry)
- Spreads vs. realised volatility forecast
- The effect of risk factors on spreads

# Imports

In [50]:
import sys
from pathlib import Path
import logging
import time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

from src.pricing_engine import PricingEngine
from src.quoting.lucic_tse import LucicTseQuotingEngine

# Setup

In [51]:
# --- Setup ---
SYMBOL = "SPY"
R = 0.045
Q = 0.012

In [52]:
# 1. Create and calibrate the PricingEngine (background thread)
pricing_engine = PricingEngine(symbol=SYMBOL, r=R, q=Q, max_expiries=4)
pricing_engine.start_background_calibration(interval_ms=2000)

# Wait for initial calibration
print("Waiting for PricingEngine calibration...")
while not pricing_engine.is_calibrated():
    time.sleep(0.5)
print("PricingEngine calibrated.")

spot = pricing_engine.get_spot()
print(f"Spot: {spot:.2f}")

2026-07-12 12:07:47,037 - src.pricing_engine - INFO - Starting initial calibration...
2026-07-12 12:07:50,134 - src.pricing_engine - INFO - Initial calibration completed successfully.
2026-07-12 12:07:50,170 - src.pricing_engine - INFO - Background calibration thread started (interval=2000ms).


Waiting for PricingEngine calibration...
PricingEngine calibrated.
Spot: 754.95


In [53]:
# 2. Build option specs (a few strikes and expiries)
svi_slices = pricing_engine.get_svi_params()
expiries = sorted(svi_slices.keys())[:2]  # use two expiries from SVI: Ensures that the volatility surface has been calibrated to the expiry
option_specs = []

for expiry in expiries:
    T = svi_slices[expiry]['T']
    strikes = [int(spot * (1 + i*0.01)) for i in range(-3, 4)]  # 6 strikes around ATM
    for strike in strikes:
        option_specs.append({
            'id': f"SPY_{strike}_{expiry}",
            'strike': float(strike),
            'expiry': expiry,
            'T': T,
            'option_type': 'call',
        })
print(f"Created {len(option_specs)} option specs.")
print(option_specs)

Created 14 option specs.
[{'id': 'SPY_732_2026-07-13', 'strike': 732.0, 'expiry': '2026-07-13', 'T': 0.0038664333123414507, 'option_type': 'call'}, {'id': 'SPY_739_2026-07-13', 'strike': 739.0, 'expiry': '2026-07-13', 'T': 0.0038664333123414507, 'option_type': 'call'}, {'id': 'SPY_747_2026-07-13', 'strike': 747.0, 'expiry': '2026-07-13', 'T': 0.0038664333123414507, 'option_type': 'call'}, {'id': 'SPY_754_2026-07-13', 'strike': 754.0, 'expiry': '2026-07-13', 'T': 0.0038664333123414507, 'option_type': 'call'}, {'id': 'SPY_762_2026-07-13', 'strike': 762.0, 'expiry': '2026-07-13', 'T': 0.0038664333123414507, 'option_type': 'call'}, {'id': 'SPY_770_2026-07-13', 'strike': 770.0, 'expiry': '2026-07-13', 'T': 0.0038664333123414507, 'option_type': 'call'}, {'id': 'SPY_777_2026-07-13', 'strike': 777.0, 'expiry': '2026-07-13', 'T': 0.0038664333123414507, 'option_type': 'call'}, {'id': 'SPY_732_2026-07-14', 'strike': 732.0, 'expiry': '2026-07-14', 'T': 0.0066061588958967535, 'option_type': 'call'}

In [54]:
# 3. Define risk factors and order flow parameters
risk_factors = [
    {
        'name': 'Vega 0-7D',
        'alpha': 10.0,
        'membership': lambda spec: spec['T'] <= 7/365,
        'weight_key': 'vega',
    },
    {
        'name': 'Vega 8-30D',
        'alpha': 5.0,
        'membership': lambda spec: 7/365 < spec['T'] <= 30/365,
        'weight_key': 'vega',
    },
    {
        'name': 'Total Delta',
        'alpha': 1.0,
        'membership': lambda spec: True,
        'weight_key': 'delta',
    },
]

order_flow_params = {
    'lambda0_a': 50 * 252,
    'lambda0_b': 50 * 252,
    'kappa_a': 0.75,
    'kappa_b': 0.75,
}


In [55]:
# 4. Create quoting engine (auto_update=True)
engine = LucicTseQuotingEngine(
    pricing_engine=pricing_engine,
    risk_factors=risk_factors,
    order_flow_params=order_flow_params,
    horizon_hours=0.5,
    risk_aversion=0.1,
    auto_update=True,
    update_interval_ms=500,
    initial_realized_vol=0.18,
    option_specs=option_specs,
)

# Wait for initial state update
print("Waiting for quoting engine initial state...")
while not engine.is_initialized():
    time.sleep(0.5)
print("Quoting engine initialized.")

2026-07-12 12:07:51,285 - src.quoting.lucic_tse - INFO - Initial realized vol set to 0.1800
2026-07-12 12:07:51,394 - src.quoting.lucic_tse - INFO - Initialised with 14 option specs.
2026-07-12 12:07:51,474 - src.quoting.lucic_tse - INFO - Performing initial synchronous state update...
2026-07-12 12:07:51,892 - src.quoting.lucic_tse - INFO - Initial state update completed.
2026-07-12 12:07:52,024 - src.quoting.lucic_tse - INFO - Starting background update thread (interval=500ms).


Waiting for quoting engine initial state...
Quoting engine initialized.


# Test: Spreads vs. Inventory 

## Calcualte bid, ask, and spread using the LucicTseEngine

In [56]:
# Vary inventory of the ATM call from -5 to +5, keep others zero
inventories = np.arange(-5, 6)
atm_spec = [s for s in option_specs if s['strike'] == int(spot)][0]
atm_id = atm_spec['id']

bid_prices = []
ask_prices = []
spreads = []
fair_values = []

for q in inventories:
    positions = {atm_id: q}
    quotes = engine.generate_quotes(option_specs, positions)
    q_quote = quotes[atm_id]
    bid_prices.append(q_quote.bid)
    ask_prices.append(q_quote.ask)
    spreads.append(q_quote.spread)
    fair_values.append(q_quote.fair_value)

## Plot
Observe the following:
- Spread is practically constant w.r.t to the inventory which agrees with theory
- Ask and Bid increases when being short (buying pressure) and decreases when being long (selling pressure) 

In [57]:
# Plot
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("Bid/Ask vs Inventory", "Spread vs Inventory"))
fig.add_trace(go.Scatter(x=inventories, y=bid_prices, mode='lines+markers', name='Bid'), row=1, col=1)
fig.add_trace(go.Scatter(x=inventories, y=ask_prices, mode='lines+markers', name='Ask'), row=1, col=1)
fig.add_trace(go.Scatter(x=inventories, y=spreads, mode='lines+markers', name='Spread'), row=2, col=1)
fig.update_yaxes(row=2, col=1, range=[0, max(spreads) * 1.2])
fig.update_layout(
    title="Quotes vs Inventory (ATM Call)",
    xaxis_title="Inventory (q)",
    yaxis_title="USD",
    height=600)
fig.show()

# Test: Spreads vs. Realised Vol Forecast

In [58]:
# List of realised vol
vol_forecasts = np.linspace(0.10, 0.40, 10)
spreads_vs_vol = []

for vol in vol_forecasts:
    engine.set_vol_forecast(vol)
    # Allow background thread to update (sleep a bit)
    time.sleep(0.2)
    quotes = engine.generate_quotes(option_specs, {})
    # Get spread of ATM call
    spread = quotes[atm_id].spread
    spreads_vs_vol.append(spread)

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=vol_forecasts, y=spreads_vs_vol, mode='lines+markers'))
fig2.update_yaxes(range=[min(spreads_vs_vol)*0.9, max(spreads_vs_vol) * 1.1])
fig2.update_layout(title="ATM Call Spread vs. Realised Vol Forecast",
                   xaxis_title="Realised Vol Forecast",
                   yaxis_title="Spread (price)")
fig2.show()

c:\Users\Danny\Desktop\Stuff\Projects\Projects - Finance\Market Making Bot\Volatility-Adaptive-Options-Market-Making-Engine\src\models\svi.py:13: RuntimeWarning: invalid value encountered in sqrt
  return np.sqrt(svi_total_variance(k, a, b, rho, m, sigma) / T)


In [59]:
# Calculate midpoints
midpoints = []
for q in inventories:
    positions = {atm_id: q}
    quotes = engine.generate_quotes(option_specs, positions)
    q_quote = quotes[atm_id]
    midpoints.append((q_quote.bid + q_quote.ask) / 2)

# Plot midpoint vs inventory
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=inventories, y=midpoints, mode='lines+markers', name='Midpoint'))
fig3.update_layout(title="Midpoint vs Inventory (ATM Call)",
                   xaxis_title="Inventory (q)",
                   yaxis_title="Midpoint Price")
fig3.show()

# Display current parameters

In [60]:
print("=== Current Engine State ===")
print(f"Number of options: {engine._N}")
print(f"Is initialized: {engine.is_initialized()}")
print(f"SSVI params: {pricing_engine.get_ssvi_params()}")
print(f"Risk A matrix shape: {engine._A.shape if engine._A is not None else 'None'}")
print(f"Theta_2 shape: {engine._Theta_2.shape if engine._Theta_2 is not None else 'None'}")

# Show some quotes for zero inventory
quotes_zero = engine.generate_quotes(option_specs, {})
print("\nSample quotes (zero inventory):")
for k, q in list(quotes_zero.items())[:3]:
    print(f"{k}: Bid={q.bid:.2f}, Ask={q.ask:.2f}, Spread={q.spread:.2f}")

=== Current Engine State ===
Number of options: 14
Is initialized: True
SSVI params: {'rho': np.float64(0.9306731543321096), 'eta': np.float64(1.9517020592017866), 'gamma': 0.5}
Risk A matrix shape: (14, 14)
Theta_2 shape: (14, 14)

Sample quotes (zero inventory):
SPY_732_2026-07-13: Bid=28.61, Ask=31.47, Spread=2.86
SPY_739_2026-07-13: Bid=25.95, Ask=28.84, Spread=2.90
SPY_747_2026-07-13: Bid=23.51, Ask=26.43, Spread=2.92


# Cleanup

In [61]:
engine.stop_background_update()
pricing_engine.stop_background_calibration()
print("Cleanup complete.")

2026-07-12 12:08:07,077 - src.quoting.lucic_tse - INFO - Stopping background update thread...
2026-07-12 12:08:08,092 - src.quoting.lucic_tse - WARNING - Background thread did not stop within 1s. It will stop on its next cycle or when the program exits. The thread reference is kept to prevent a new thread from starting.
2026-07-12 12:08:08,184 - src.quoting.lucic_tse - INFO - Background update thread stopped (or marked for stopping).
2026-07-12 12:08:08,434 - src.pricing_engine - INFO - Stopping background calibration thread...
2026-07-12 12:08:09,457 - src.pricing_engine - INFO - Background calibration thread stopped.


Cleanup complete.
